# Exp 17 — Просмотр результатов eval на реальных данных

Три модели для сравнения (все с одним энкодером bge-m3, чтобы изолировать эффект обучения):

| Модель | Обучение | Назначение |
|--------|----------|------------|
| **Baseline** | нет (raw bge-m3) | zero-shot bge-m3 |
| **v14 GNN** | Magellan only | без русских view-pairs |
| **v17 GNN** | Magellan + RuViews | с русскими view-pairs |

Все три используют один и тот же кэш реального графа `output/07_real_data_test_bge-m3/`. Baseline-эмбеддинги одинаковы во всех результатах (это просто `graph["row"].x` нормированные), поэтому достаточно прочитать их из любого `results.json`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODEL_TAG = "bge-m3"
LOSS = "ntxent"  # "bce" или "ntxent"
suffix = "_bce" if LOSS == "bce" else ""

# Пути к eval-результатам каждой модели
UNSUP_V17 = Path("../../output") / f"07_real_data_test_{MODEL_TAG}" / "results.json"
UNSUP_V14 = Path("../../output") / f"07_real_data_test_{MODEL_TAG}_v14" / "results.json"

SUP_V17 = (Path("../../output") / MODEL_TAG /
           f"v17_views_gat{suffix}_model_real_trusted_eval.json")
SUP_V14 = (Path("../../output") / MODEL_TAG /
           f"v14_mrl_gat{suffix}_model_real_trusted_eval.json")

print("Unsupervised:")
print("  v17:", UNSUP_V17, "→", "OK" if UNSUP_V17.exists() else "НЕТ")
print("  v14:", UNSUP_V14, "→", "OK" if UNSUP_V14.exists() else "НЕТ")
print("Supervised:")
print("  v17:", SUP_V17, "→", "OK" if SUP_V17.exists() else "НЕТ")
print("  v14:", SUP_V14, "→", "OK" if SUP_V14.exists() else "НЕТ")

## 1. Unsupervised метрики (exp 07)

Категория, reciprocal nearest-neighbor, confidence separation. Сравнение GNN vs raw rubert/bge baseline.

In [ ]:
unsup_v17 = json.loads(UNSUP_V17.read_text()) if UNSUP_V17.exists() else {}
unsup_v14 = json.loads(UNSUP_V14.read_text()) if UNSUP_V14.exists() else {}

def _row(name: str, r: dict) -> dict:
    return {
        "method": name,
        "category consistency": r["category_consistency"]["precision"],
        "reciprocal rate":     r["reciprocal"]["rate"],
        "confidence gap":      r.get("confidence", {}).get("mean_gap"),
        "mean top-1 sim":      r.get("confidence", {}).get("mean_top1_sim"),
    }

rows = []
# Baseline берём из любого доступного results.json — он одинаковый
base_src = unsup_v17 or unsup_v14
if base_src and "Baseline" in base_src:
    rows.append(_row("Baseline (bge-m3)", base_src["Baseline"]))
if unsup_v14 and "GNN" in unsup_v14:
    rows.append(_row("v14 GNN (Magellan only)", unsup_v14["GNN"]))
if unsup_v17 and "GNN" in unsup_v17:
    rows.append(_row("v17 GNN (+ RuViews)", unsup_v17["GNN"]))

df_unsup = pd.DataFrame(rows).set_index("method")
df_unsup.style.format("{:.3f}", na_rep="—").background_gradient(cmap="Greens", axis=0)

In [ ]:
metrics = ["category consistency", "reciprocal rate"]
colors = {"Baseline (bge-m3)": "#bbb",
          "v14 GNN (Magellan only)": "#f4a261",
          "v17 GNN (+ RuViews)": "#5b9bd5"}

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(metrics))
n = len(df_unsup.index)
w = 0.8 / max(n, 1)
for i, method in enumerate(df_unsup.index):
    vals = df_unsup.loc[method, metrics].values.astype(float)
    offset = (i - (n - 1) / 2) * w
    bars = ax.bar(x + offset, vals, w, label=method, color=colors.get(method, "#888"))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("score")
ax.set_title("Unsupervised: Baseline vs v14 vs v17")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, max(1.0, df_unsup[metrics].values.max() * 1.15))
plt.tight_layout()
plt.show()

### Per-category breakdown

In [ ]:
per_cat = {}

def _accum(label: str, r: dict):
    pc = r.get("category_consistency", {}).get("per_category", {})
    for cat, v in pc.items():
        per_cat.setdefault(cat, {})
        per_cat[cat][f"{label} P"] = v.get("precision", 0)
        per_cat[cat][f"{label} n"] = v.get("total", 0)

if base_src and "Baseline" in base_src:
    _accum("Base", base_src["Baseline"])
if unsup_v14 and "GNN" in unsup_v14:
    _accum("v14", unsup_v14["GNN"])
if unsup_v17 and "GNN" in unsup_v17:
    _accum("v17", unsup_v17["GNN"])

df_cat = pd.DataFrame(per_cat).T
# Сортируем по самому "богатому" n среди столбцов
n_cols = [c for c in df_cat.columns if c.endswith(" n")]
if n_cols:
    df_cat = df_cat.sort_values(n_cols[0], ascending=False)
df_cat.head(20)

## 2. Supervised метрики (exp 15)

F1/P/R/ROC-AUC/AP на gold (398 пар, ручная разметка) и silver (~26K, LLM-метки).

In [ ]:
sup_v17 = json.loads(SUP_V17.read_text()) if SUP_V17.exists() else {}
sup_v14 = json.loads(SUP_V14.read_text()) if SUP_V14.exists() else {}

def _flatten(label: str, sup: dict, method_key: str) -> list[dict]:
    out = []
    for tier, r in sup.items():
        m = r.get(method_key, {})
        if not m:
            continue
        out.append({
            "tier":      tier,
            "model":     label,
            "n_pairs":   m.get("n_pairs", 0),
            "n_pos":     m.get("n_pos", 0),
            "threshold": m.get("threshold"),
            "F1":        m.get("f1"),
            "Precision": m.get("precision"),
            "Recall":    m.get("recall"),
            "ROC-AUC":   m.get("roc_auc"),
            "AP":        m.get("avg_precision"),
        })
    return out

rows = []
# Baseline один и тот же в обоих файлах — берём из любого
if sup_v17:
    rows += _flatten("Baseline (bge-m3)", sup_v17, "Baseline")
elif sup_v14:
    rows += _flatten("Baseline (bge-m3)", sup_v14, "Baseline")
if sup_v14:
    rows += _flatten("v14 GNN (Magellan only)", sup_v14, "GNN")
if sup_v17:
    rows += _flatten("v17 GNN (+ RuViews)", sup_v17, "GNN")

df_sup = pd.DataFrame(rows).set_index(["tier", "model"])
df_sup.style.format({
    "threshold": "{:.3f}", "F1": "{:.3f}", "Precision": "{:.3f}",
    "Recall": "{:.3f}", "ROC-AUC": "{:.3f}", "AP": "{:.3f}",
    "n_pairs": "{:,.0f}", "n_pos": "{:,.0f}",
}, na_rep="—").background_gradient(
    cmap="Greens", subset=["F1", "Precision", "Recall", "ROC-AUC", "AP"], axis=0,
)

In [ ]:
tiers = sorted({t for t, _ in df_sup.index})
metrics_cols = ["F1", "Precision", "Recall", "ROC-AUC", "AP"]
model_order = ["Baseline (bge-m3)", "v14 GNN (Magellan only)", "v17 GNN (+ RuViews)"]
model_colors = {"Baseline (bge-m3)": "#bbb",
                "v14 GNN (Magellan only)": "#f4a261",
                "v17 GNN (+ RuViews)": "#5b9bd5"}

fig, axes = plt.subplots(1, len(tiers), figsize=(6 * len(tiers), 4.5), squeeze=False)
for j, tier in enumerate(tiers):
    ax = axes[0, j]
    x = np.arange(len(metrics_cols))
    present = [m for m in model_order if (tier, m) in df_sup.index]
    n = len(present)
    w = 0.8 / max(n, 1)
    for i, model in enumerate(present):
        vals = df_sup.loc[(tier, model), metrics_cols].values.astype(float)
        offset = (i - (n - 1) / 2) * w
        bars = ax.bar(x + offset, vals, w, label=model, color=model_colors[model])
        for bar, v in zip(bars, vals):
            if v == v:
                ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.2f}",
                        ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(metrics_cols, rotation=0)
    any_row = next(iter(present), None)
    if any_row:
        n_pairs = df_sup.loc[(tier, any_row), "n_pairs"]
        n_pos = df_sup.loc[(tier, any_row), "n_pos"]
        ax.set_title(f"{tier}  (n={n_pairs}, +={n_pos})")
    else:
        ax.set_title(tier)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right", fontsize=7)
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### Threshold sweep — F1 vs порог

In [ ]:
fig, axes = plt.subplots(1, len(tiers), figsize=(6 * len(tiers), 4.5), squeeze=False)
sources = [
    ("Baseline (bge-m3)", sup_v17 or sup_v14, "Baseline"),
    ("v14 GNN (Magellan only)", sup_v14, "GNN"),
    ("v17 GNN (+ RuViews)", sup_v17, "GNN"),
]

for j, tier in enumerate(tiers):
    ax = axes[0, j]
    for label, sup, method in sources:
        if not sup:
            continue
        sweep = sup.get(tier, {}).get(method, {}).get("sweep_f1", [])
        if not sweep:
            continue
        sweep_arr = np.asarray(sweep, dtype=object)
        # sweep может быть: list of (thr, f1) → 2d; list of f1 → 1d; скаляр → 0d (пропускаем)
        if sweep_arr.ndim == 0 or sweep_arr.size == 0:
            continue
        if sweep_arr.ndim == 2:
            arr = sweep_arr.astype(float)
            xs, ys = arr[:, 0], arr[:, 1]
        else:
            ys = np.asarray(sweep_arr, dtype=float).ravel()
            xs = np.linspace(0, 1, len(ys))
        color = model_colors[label]
        ax.plot(xs, ys, label=label, color=color, lw=2)
        best_thr = sup[tier][method].get("threshold")
        best_f1 = sup[tier][method].get("f1")
        if best_thr is not None and best_f1 is not None:
            ax.axvline(best_thr, color=color, ls="--", alpha=0.3)
            ax.scatter([best_thr], [best_f1], color=color, zorder=5)
    ax.set_xlabel("threshold (cosine sim)")
    ax.set_ylabel("F1")
    ax.set_title(tier)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Per-strategy breakdown (silver/silver+)

Раскладка по способу формирования меток (например: exact_match / fuzzy / nomatch_high_sim и т.п.).

In [ ]:
strat_rows = []
for label, sup, method in sources:
    if not sup:
        continue
    for tier, r in sup.items():
        per_strat = r.get(method, {}).get("per_strategy", {})
        for strat, m in per_strat.items():
            strat_rows.append({
                "tier": tier, "model": label, "strategy": strat,
                "n": m.get("n", 0), "n_pos": m.get("n_pos", 0),
                "F1": m.get("f1"),
            })

if strat_rows:
    df_strat = pd.DataFrame(strat_rows)
    pivot = df_strat.pivot_table(
        index=["tier", "strategy", "n", "n_pos"],
        columns="model", values="F1",
    ).reset_index()
    pivot = pivot.sort_values(["tier", "n"], ascending=[True, False])
    fmt = {c: "{:.3f}" for c in pivot.columns if c not in ("tier", "strategy", "n", "n_pos")}
    display(pivot.style.format(fmt, na_rep="—"))
else:
    print("per_strategy не найден — поле было пустым")

## 3. Сводка одной таблицей

In [ ]:
print(f"=== Real-data eval: bge-m3 / {LOSS} ===\n")
print("Unsupervised:")
for method in df_unsup.index:
    cc = df_unsup.loc[method, "category consistency"]
    rr = df_unsup.loc[method, "reciprocal rate"]
    print(f"  {method:30s}  category={cc:.3f}   reciprocal={rr:.3f}")

print("\nSupervised:")
for (tier, model), row in df_sup.iterrows():
    f1 = row["F1"]; p = row["Precision"]; r = row["Recall"]
    auc = row["ROC-AUC"]; ap = row["AP"]
    print(f"  {tier:18s} / {model:28s}  "
          f"F1={f1:.3f}  P={p:.3f}  R={r:.3f}  AUC={auc:.3f}  AP={ap:.3f}")